## Assignment 3 Graded Assignment - Building BPE

###  **Objective**

Implement the Byte Pair Encoding (BPE) tokenizer from scratch. You will write the code to **train** the tokenizer on a corpus to learn merge rules and then write the code to **tokenize** new text using those learned rules.

###  **Assignment Structure**

This assignment will contain:

1.  **Setup Block:** Imports and a small sample `corpus` string to train on.
2.  **Part 1: Pre-tokenization & Initial Vocab:** Functions to process raw text into a BPE-ready format.
3.  **Part 2: BPE Core Logic:** The helper functions to find the most frequent pair and to merge that pair.
4.  **Part 3: The Training Loop:** A function that combines all pieces to generate the vocab and merge rules.
5.  **Part 4: The Tokenizer:** A `BPETokenizer` class that uses the trained files to encode new, unseen text.

In [10]:
import re
import json
import collections
from tqdm import tqdm # A nice progress bar for our training loop

# --- Configuration for testing ---
# We will learn 50 new "merge" rules.
NUM_MERGES = 50

# --- Sample Corpus ---
# This is the raw text we will use to train our tokenizer.
corpus = """
This is a simple corpus.
It is used to demonstrate the Byte Pair Encoding algorithm.
BPE is a subword tokenization strategy.
It starts with a vocabulary of individual characters
and iteratively merges the most frequent adjacent pair of symbols.
This helps in handling rare words and reducing vocabulary size.
"""

print("="*50)
print("RUNNING BONUS ASSIGNMENT 3: BPE TOKENIZER")
print(f"Training with NUM_MERGES = {NUM_MERGES}")
print("="*50)

RUNNING BONUS ASSIGNMENT 3: BPE TOKENIZER
Training with NUM_MERGES = 50


-----

### **Part 1: Pre-tokenization and Initial Vocab**

**Instructions:** Before we can count pairs, we need to process our raw text. We will split the corpus into words, count their frequencies, and format them for BPE. Our format will be a dictionary where keys are tuples of characters representing a word (e.g., `('h', 'e', 'l', 'l', 'o', '</w>')`) and values are their frequencies. The special `</w>` token marks the end of a word.

In [11]:
print("\n--- Part 1: Pre-tokenization and Initial Vocab ---")

def get_word_freqs(text):
    """
    Splits the text into words and counts their frequency.
    We use a simple regex to find words.
    """
    # 1. Find all words (sequences of letters) and convert to lowercase
    words = re.findall(r'\b\w+\b', text.lower())

    # 2. Count the frequency of each word
    # (Hint: use collections.Counter)
    return collections.Counter(words)

def initialize_bpe_corpus(word_freqs):
    """
    Converts word frequencies into the BPE-ready format
    and creates the initial character vocabulary.
    """
    bpe_corpus = collections.defaultdict(int)
    initial_vocab = set()

    for word, freq in word_freqs.items():
        # 1. Create the tuple format for the word
        # e.g., "hello" -> ('h', 'e', 'l', 'l', 'o', '</w>')
        word_tuple = tuple(word) + ('</w>',)

        # 2. Store its frequency
        bpe_corpus[word_tuple] = freq

        # 3. Add all individual characters to the initial vocab
        for char in word_tuple:
            initial_vocab.add(char)

    return bpe_corpus, sorted(list(initial_vocab))

# --- Verification for Part 1 ---
print("--- Running Verification for Part 1 ---")
try:
    test_text = "hello world hello"
    test_freqs = get_word_freqs(test_text)
    assert test_freqs == {'hello': 2, 'world': 1}
    print(" (1a) get_word_freqs is correct.")

    bpe_corpus, vocab = initialize_bpe_corpus(test_freqs)
    expected_corpus = {('h', 'e', 'l', 'l', 'o', '</w>'): 2, ('w', 'o', 'r', 'l', 'd', '</w>'): 1}
    assert bpe_corpus == expected_corpus
    print(" (1b) bpe_corpus is correct.")
    assert 'h' in vocab and '</w>' in vocab and len(vocab) == 8
    print(" (1c) initial_vocab is correct.")
    print(" Part 1 seems correct!")
except Exception as e:
    print(f" Part 1 Failed: {e}")


--- Part 1: Pre-tokenization and Initial Vocab ---
--- Running Verification for Part 1 ---
 (1a) get_word_freqs is correct.
 (1b) bpe_corpus is correct.
 (1c) initial_vocab is correct.
 Part 1 seems correct!


#### **Questions for Part 1**

**Q1: Conceptual Check**
What is the primary purpose of adding the special `</w>` token at the end of each word?

  * (A) To mark the start of a new word.
  * (B) To ensure all words have the same length.
  * (C) To distinguish between a character sequence *inside* a word (like "es") and one at the *end* of a word (like "es" in "files").
  * (D) To increase the total vocabulary size.


**Q2: Value Check**
Run the two functions you just wrote on the main `corpus` provided in the Setup block. What is the frequency of the word tuple `('i', 's', '</w>')`?
*(Enter the answer as an integer)*.




In [12]:
print("\n--- Q2 ---")
word_freqs_q2 = get_word_freqs(corpus)
bpe_corpus_q2, _ = initialize_bpe_corpus(word_freqs_q2)
freq_is = bpe_corpus_q2.get(('i', 's', '</w>'), 0)
print(f"Frequency of ('i', 's', '</w>'): {freq_is}")


--- Q2 ---
Frequency of ('i', 's', '</w>'): 3


### **Part 2: BPE Core Logic**

**Instructions:** Now we need to implement the two core helper functions for the BPE training loop:

1.  `get_pair_stats`: This function scans the `bpe_corpus` and counts the frequency of all adjacent symbol pairs.
2.  `merge_pair`: This function takes the most frequent pair and "merges" it into a new symbol (e.g., `('t', 'h')` becomes `'th'`) throughout the entire `bpe_corpus`.

In [13]:
print("\n--- Part 2: BPE Core Logic ---")

def get_pair_stats(bpe_corpus):
    """Counts the frequency of all adjacent symbol pairs."""
    stats = collections.defaultdict(int)
    for word_tuple, freq in bpe_corpus.items():
        # Iterate over all adjacent pairs in the word_tuple
        for i in range(len(word_tuple) - 1):
            # 1. Get the pair
            pair = (word_tuple[i], word_tuple[i + 1])
            # 2. Increment its count by the word's frequency
            stats[pair] += freq
    return stats

def merge_pair(best_pair, bpe_corpus):
    """Merges the 'best_pair' into a new symbol in the corpus."""
    new_bpe_corpus = collections.defaultdict(int)
    new_symbol = "".join(best_pair) # e.g., ('t', 'h') -> 'th'

    for word_tuple, freq in bpe_corpus.items():
        new_word_tuple = []
        i = 0
        while i < len(word_tuple):
            # 1. Check if we've found the 'best_pair' at this position
            if i < len(word_tuple) - 1 and (word_tuple[i], word_tuple[i+1]) == best_pair:
                # 2. If yes, append the new_symbol
                new_word_tuple.append(word_tuple[i] + word_tuple[i + 1])
                # 3. Skip over both original symbols
                i += 2
            else:
                # 4. If no, just append the current symbol
                new_word_tuple.append(word_tuple[i])
                # 5. Move to the next symbol
                i += 1

        # Add the newly merged word_tuple to the new corpus
        new_bpe_corpus[tuple(new_word_tuple)] = freq

    return new_bpe_corpus

# --- Verification for Part 2 ---
print("--- Running Verification for Part 2 ---")
try:
    test_corpus = {('h', 'e', 'l', 'l', 'o', '</w>'): 2, ('w', 'o', 'r', 'l', 'd', '</w>'): 1}
    stats = get_pair_stats(test_corpus)
    assert stats[('l', 'l')] == 2
    assert stats[('o', '</w>')] == 2
    assert stats[('w', 'o')] == 1
    print(" (2a) get_pair_stats is correct.")

    new_corpus = merge_pair(best_pair=('l', 'l'), bpe_corpus=test_corpus)
    expected_corpus = {('h', 'e', 'll', 'o', '</w>'): 2, ('w', 'o', 'r', 'l', 'd', '</w>'): 1}
    assert new_corpus == expected_corpus
    print(" (2b) merge_pair is correct.")

    # Check that the merged pair is gone
    stats_after_merge = get_pair_stats(new_corpus)
    assert ('l', 'l') not in stats_after_merge
    assert ('e', 'll') in stats_after_merge
    print(" (2c) Merge was successful.")
    print(" Part 2 seems correct!")
except Exception as e:
    print(f" Part 2 Failed: {e}")


--- Part 2: BPE Core Logic ---
--- Running Verification for Part 2 ---
 (2a) get_pair_stats is correct.
 (2b) merge_pair is correct.
 (2c) Merge was successful.
 Part 2 seems correct!


#### **Questions for Part 2**

**Q3: Conceptual Check**
In `get_pair_stats`, if a word appears 5 times (`freq=5`) and contains the pair `('t', 'h')` twice, how much will the total count for `('t', 'h')` be incremented by that word?

  * (A) 2
  * (B) 5
  * (C) 7
  * (D) 10



**Q4: Value Check**
Using the `test_corpus` from the verification block, what is the frequency of the pair `('l', 'd')`?
*(Enter the answer as an integer)*.



In [14]:
print("\n--- Q4 ---")
test_corpus_q4 = {('h', 'e', 'l', 'l', 'o', '</w>'): 2, ('w', 'o', 'r', 'l', 'd', '</w>'): 1}
stats_q4 = get_pair_stats(test_corpus_q4)
freq_ld = stats_q4.get(('l', 'd'), 0)
print(f"Frequency of ('l', 'd'): {freq_ld}")


--- Q4 ---
Frequency of ('l', 'd'): 1


### **Part 3: The Full Training Function**

**Instructions:** Now, let's combine all the pieces into a `train` function. This function will initialize the corpus, then loop `NUM_MERGES` times, each time finding the best pair, merging it, and saving the merge rule.

In [15]:
print("\n--- Part 3: The Full Training Function ---")

def train_bpe(raw_corpus, num_merges):
    """Trains a BPE tokenizer on a raw text corpus."""
    word_freqs = get_word_freqs(raw_corpus)
    bpe_corpus, vocab = initialize_bpe_corpus(word_freqs)
    merges = []

    for i in tqdm(range(num_merges), desc="Training BPE"):
        stats = get_pair_stats(bpe_corpus)
        if not stats:
            break

        # Use max with a key to find the most frequent pair.
        # This matches the hint provided in the notebook.
        best_pair = max(stats, key=stats.get)

        merges.append(best_pair)
        vocab.append("".join(best_pair))
        bpe_corpus = merge_pair(best_pair, bpe_corpus)

    final_vocab = {token: idx for idx, token in enumerate(vocab)}
    final_merges = {pair: idx for idx, pair in enumerate(merges)}

    return final_vocab, final_merges

# --- Verification for Part 3 ---
print("--- Running Verification for Part 3 ---")
try:
    test_corpus_full = "low lower newest widest low"
    vocab, merges = train_bpe(test_corpus_full, num_merges=10)
    assert merges[('l', 'o')] == 0
    assert merges[('lo', 'w')] == 1
    assert merges[('e', 's')] == 2
    assert merges[('es', 't')] == 3
    assert merges[('est', '</w>')] == 4
    print(" (3a) Merge order seems correct.")
    assert "low</w>" in vocab
    assert "est</w>" in vocab
    print(" (3b) Final vocab contains merged tokens.")
    print(" Part 3 seems correct!")
except Exception as e:
    print(f" Part 3 Failed: {e}")


--- Part 3: The Full Training Function ---
--- Running Verification for Part 3 ---


Training BPE: 100%|██████████| 10/10 [00:00<00:00, 36157.79it/s]

 Part 3 Failed: 


#### **Questions for Part 3**

**Q5: Conceptual Check**
Why is it critical that `final_merges` stores the *order* (rank) of the merges, and not just a `set` of valid merges?

  * (A) To save memory; an integer rank is smaller than a string.
  * (B) During tokenization, merges must be applied in the *exact same order* they were learned to get the correct tokens.
  * (C) It's not critical; a `set` would also work.
  * (D) To ensure the `vocab` and `merges` have the same number of items.



**Q6: Value Check**
Run `train_bpe` on the main `corpus` from the Setup block with `NUM_MERGES = 10`. What is the **5th** merge rule learned (i.e., the pair at index 4 in the `merges` list)?

  * (A) ('i', 'n')
  * (B) ('i', 's')
  * (C) ('e', 'n')
  * (D) ('t', '</w>')



In [16]:
print("\n--- Q6 ---")
# We use the 3rd returned value, the raw 'merges' list
_, merges_dict_q6 = train_bpe(corpus, num_merges=10)

# Get the list of merges from the dictionary keys
merges_list_q6 = list(merges_dict_q6.keys())

if len(merges_list_q6) > 4:
    fifth_merge = merges_list_q6[4]
    print(f"The 5th merge rule (index 4) is: {fifth_merge}")
else:
    print("Training produced fewer than 5 merges.")


--- Q6 ---


Training BPE: 100%|██████████| 10/10 [00:00<00:00, 4731.84it/s]

The 5th merge rule (index 4) is: ('t', '</w>')


### **Part 4: The Tokenizer (Inference)**

**Instructions:** We're almost done\! Now, create a `BPETokenizer` class. It will load the `vocab` and `merges` you just trained. Its `tokenize` method will take a new string, pre-tokenize it, and then **iteratively apply the learned merges in the correct order** to get the final list of token IDs.

In [17]:
print("\n--- Part 4: The Tokenizer (Inference) ---")

class BPETokenizer:
    def __init__(self, vocab, merges):
        self.vocab = vocab
        self.merges = merges
        self.unk_token_id = self.vocab.get('<UNK>', 0)

    def tokenize(self, text):
        words = re.findall(r'\b\w+\b', text.lower())
        all_token_ids = []

        for word in words:
            symbols = list(word) + ['</w>']
            while True:
                pairs = [(symbols[i], symbols[i+1]) for i in range(len(symbols) - 1)]
                best_pair_to_merge = None
                min_rank = float('inf')

                for pair in pairs:
                    if pair in self.merges and self.merges[pair] < min_rank:
                        min_rank = self.merges[pair]
                        best_pair_to_merge = pair

                if best_pair_to_merge is None:
                    break

                new_symbols = []
                i = 0
                while i < len(symbols):
                    if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == best_pair_to_merge:
                        new_symbols.append("".join(best_pair_to_merge))
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                symbols = new_symbols

            for symbol in symbols:
                token_id = self.vocab.get(symbol, self.unk_token_id)
                all_token_ids.append(token_id)
        return all_token_ids

# --- Verification for Part 4 ---
print("--- Running Verification for Part 4 ---")
try:
    test_corpus_full = "low lower newest widest low"
    vocab, merges = train_bpe(test_corpus_full, num_merges=10)
    tokenizer = BPETokenizer(vocab, merges)
    ids = tokenizer.tokenize("lowest")
    expected_ids = [vocab["low"], vocab["est</w>"]]
    assert ids == expected_ids
    print(" (4a) 'lowest' tokenized correctly.")
    ids = tokenizer.tokenize("wider")
    expected_ids = [vocab["w"], vocab.get("i", 0), vocab.get("d", 0), vocab["e"], vocab.get("r", 0), vocab["</w>"]]
    assert ids == expected_ids
    print(" (4b) OOV word 'wider' tokenized correctly.")
    print(" Part 4 seems correct! Assignment complete.")
except Exception as e:
    print(f" Part 4 Failed: {e}")


--- Part 4: The Tokenizer (Inference) ---
--- Running Verification for Part 4 ---


Training BPE: 100%|██████████| 10/10 [00:00<00:00, 36889.22it/s]

 (4a) 'lowest' tokenized correctly.
 (4b) OOV word 'wider' tokenized correctly.
 Part 4 seems correct! Assignment complete.


#### **Questions for Part 4**

**Q7: Conceptual Check**
In the `tokenize` method, if the word is "newer", and our `merges` dictionary contains both `('n', 'e')` at rank 5 and `('e', 'w')` at rank 3, which merge will be applied *first*?

  * (A) `('n', 'e')` -\> "ne"
  * (B) `('e', 'w')` -\> "ew"
  * (C) It's non-deterministic.
  * (D) Neither, it will merge `('e', 'r')` first.



**Q8: Value Check**
First, train a tokenizer on the main `corpus` with `NUM_MERGES = 50`.

In [18]:
# Code for Q8
vocab, merges = train_bpe(corpus, num_merges=50)
tokenizer = BPETokenizer(vocab, merges)
print(tokenizer.tokenize("This is a simple vocabulary"))

Training BPE: 100%|██████████| 50/50 [00:00<00:00, 6296.81it/s]

[44, 29, 37, 75, 16, 12, 27, 65]


What is the list of token IDs returned for the text "This is a simple vocabulary"?

(A) [44, 29, 37, 75, 16, 12, 27, 65]

(B) [44, 29, 37, 78]

(C) [44, 29, 37, 75, 16, 12, 27, 66, 67]

(D) [17, 8, 9, 15, 29, 9, 15, 37, 15, 9, 13, ...]

